In [1]:
import os
import sys
sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from app.retrieval.hybrid import HybridRetriever
from app.retrieval.bm25 import BM25Retriever
from app.retrieval.faiss_retriever import FaissRetriever
from app.conversation.recommender import AssessmentRecommender

In [3]:
df = pd.read_csv("../data/processed/catalog_processed.csv")
embeddings = np.load("../data/embeddings/catalog_embeddings.npy")
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
bm25 = BM25Retriever(df)
faiss = FaissRetriever(
    df=df,
    embeddings=embeddings
)
hybrid = HybridRetriever(
    bm25=bm25,
    faiss=faiss
)
llm = AssessmentRecommender()

In [5]:
class ChatAgent:
    def __init__(
        self,
        retriever,
        embedder,
        llm
    ):
        self.retriever = retriever
        self.embedder = embedder
        self.llm = llm

In [6]:
class ChatAgent:
    def __init__(
        self,
        retriever,
        embedder,
        llm
    ):
        self.retriever = retriever
        self.embedder = embedder
        self.llm = llm
        
    def chat(self, query):
        print(query)

In [7]:
agent = ChatAgent(
    hybrid,
    embedder,
    llm
)

agent.chat(
    "Need Python assessment"
)

Need Python assessment


In [23]:
def is_vague(query):
    query = query.lower()
    # Too short
    if len(query.split()) <= 1:
        return True
    role_words = [
        "engineer",
        "developer",
        "analyst",
        "manager",
        "graduate",
        "intern",
        "python",
        "java",
        "sql",
        "sales",
        "marketing",
        "finance",
        "cloud",
        "data",
        "software",
        "security",
        "leader"
    ]

    for word in role_words:
        if word in query:
            return False

    return True

In [24]:
is_vague("assessment")

True

In [25]:
is_vague("graduate java engineer")

False

In [26]:
is_vague("Need assessment")

True

In [28]:
class ChatAgent:

    def __init__(self, retriever, embedder, llm):
        self.retriever = retriever
        self.embedder = embedder
        self.llm = llm

    def chat(self, query):

        # Step 1: Ask for clarification if needed
        if is_vague(query):
            return {
                "reply": "Could you tell me the role, seniority level, or important skills you're hiring for?",
                "recommendations": []
            }

        # Step 2: Embed the query
        query_embedding = self.embedder.encode([query])

        # Step 3: Retrieve top assessments
        results = self.retriever.search(
            query=query,
            query_embedding=query_embedding,
            top_k=5
        )

        # Step 4: Build text for LLM
        assessment_text = ""

        for result in results:
            assessment_text += f"""
Name: {result['name']}
Assessment Type: {result['assessment_type']}
Job Level: {result['job_level']}
Duration: {result['duration']}
URL: {result['url']}

"""

        # Step 5: Ask the LLM
        explanation = self.llm.recommend(
            query=query,
            assessments=assessment_text
        )

        # Step 6: Return everything
        return {
            "reply": explanation,
            "recommendations": results
        }

In [29]:
agent = ChatAgent(
    hybrid,
    embedder,
    llm
)

In [30]:
agent.chat("assessment")

{'reply': "Could you tell me the role, seniority level, or important skills you're hiring for?",
 'recommendations': []}

In [31]:
agent.chat("graduate java engineer")

{'reply': "Here's a brief explanation of why each assessment is relevant for a Java graduate engineer:\n\n1. **Java 8 (New)**: Assesses understanding of Java 8 features, ensuring the candidate has adapted to the latest language updates and can apply them in their work.\n\n2. **Java Design Patterns (New)**: Evaluates the candidate's knowledge of design patterns specific to Java, demonstrating their ability to write efficient, maintainable, and scalable code.\n\n3. **Core Java (Advanced Level) (New)**: Tests the candidate's advanced understanding of core Java concepts, including multithreading, networking, and security, to assess their expertise as an individual contributor.\n\n4. **Java Platform Enterprise Edition 7 (Java EE 7)**: Assesses the candidate's knowledge of Java EE 7 features, ensuring they have hands-on experience with enterprise-level development, including application servers, web services, and EJBs.\n\n5. **Graduate Scenarios Narrative Report**: Evaluates the candidate's 

In [32]:
def display_response(response):

    print("=" * 90)
    print("AI Recommendation")
    print("=" * 90)

    print(response["reply"])

    print("\nRecommended Assessments")
    print("-" * 90)

    for i, rec in enumerate(response["recommendations"], start=1):

        print(f"{i}. {rec['name']}")
        print(f"   Type      : {rec['assessment_type']}")
        print(f"   Job Level : {rec['job_level']}")
        print(f"   Duration  : {rec['duration']}")
        print(f"   Score     : {rec['score']:.3f}")
        print(f"   URL       : {rec['url']}")
        print()

In [33]:
response = agent.chat("graduate java engineer")
display_response(response)

AI Recommendation
Here's a brief explanation of why each assessment is relevant for a graduate Java engineer:

1. **Java 8 (New)**: This assessment tests the candidate's fundamental knowledge of Java 8, ensuring they're proficient in its key features and syntax.

2. **Java Design Patterns (New)**: As a developer, understanding design patterns is crucial for writing efficient, scalable, and maintainable code. This assessment evaluates their ability to apply design principles to real-world scenarios.

3. **Core Java (Advanced Level) (New)**: Assessing advanced-level knowledge of Core Java helps evaluate the candidate's in-depth understanding of Java fundamentals, including advanced topics like multithreading, exceptions, and collections.

4. **Java Platform Enterprise Edition 7 (Java EE 7)**: Given that many enterprises still use Java EE for their applications, this assessment ensures the graduate understands the key features, frameworks, and best practices required to develop robust ent